# TrazaRed HUV — Semana 8: proyecto dockerizado, bloqueo de usuarios, dataset e imágenes (PACS)

Este notebook sigue el mismo camino de la clase (semana 7 → semana 8), pero aplicado a **nuestro**
proyecto de remisiones. Se ejecuta de arriba hacia abajo, en orden.

```
                          navegador  →  http://localhost:8080
                                   │
                         ┌─────────▼──────────┐
                         │  front  (nginx)    │  interfaz gráfica (frontend/)
                         └────┬──────────┬────┘
                     /api/*   │          │   /fhir/*
                   ┌──────────▼───┐  ┌───▼──────────────────┐
                   │ api (FastAPI)│  │ hapi-fhir + fhir-db  │  interoperabilidad FHIR R4
                   └───┬──────┬───┘  └──────────────────────┘
                       │      │
              ┌────────▼─┐ ┌──▼────────────────┐ ┌───────────────────┐
              │ db       │ │ mongo             │ │ orthanc (PACS)    │
              │ Postgres │ │ gestiones_contacto│ │ imágenes DICOM    │
              └──────────┘ └───────────────────┘ └───────────────────┘
```

**Tareas de la semana 8 que quedan resueltas aquí:**
1. Montar la base de datos del proyecto final (dataset de remisiones con 3 grupos: efectiva, fallida y borderline).
2. Control de acceso: bloqueo del usuario al **3er intento fallido** de contraseña; cada intento queda en `auditoria` y
   **solo el administrador** puede desbloquear.

Y además: todo el proyecto corre en Docker (7 contenedores), con interfaz gráfica por rol, servidor de imágenes
médicas **PACS (Orthanc)** y un visor con brillo, contraste, zoom y desplazamiento, como en el cuaderno de la clase.

---
## 0. Rutas del proyecto
El notebook vive en `notebooks/`; la raíz del proyecto se detecta sola (es la carpeta que tiene `db/schema.sql`).

In [1]:
from pathlib import Path

RAIZ = Path.cwd()
while not (RAIZ / "db" / "schema.sql").exists() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
COMPOSE = RAIZ / "docker" / "docker-compose.yml"
print("Raíz del proyecto:", RAIZ)
print("docker-compose.yml:", COMPOSE, "->", "OK" if COMPOSE.exists() else "NO EXISTE")

Raíz del proyecto: c:\Users\Acer\OneDrive\Escritorio\CARPETAS\Octavo semestre\Salud digital\TrazaRedHUV_semana8_v2
docker-compose.yml: c:\Users\Acer\OneDrive\Escritorio\CARPETAS\Octavo semestre\Salud digital\TrazaRedHUV_semana8_v2\docker\docker-compose.yml -> OK


---
## 1. Verificar Docker
Docker Desktop tiene que estar **abierto y corriendo** antes de seguir.

In [2]:
!docker --version
!docker compose version

Docker version 29.7.2, build a7dcaa6
Docker Compose version v5.5.0


---
## 2. Variables de entorno (`pass.env`)

El contenedor `api` lee `pass.env` (en la raíz del proyecto). De ahí toma:
- `SECRET_KEY` → firma de los JWT (HS256).
- `DEMO_ADMIN_PASSWORD`, `DEMO_MEDICO_PASSWORD`, `DEMO_EPS_PASSWORD`, `DEMO_PACIENTE_PASSWORD` → con estas la API
  crea sola los usuarios de prueba la primera vez que arranca (la base del contenedor empieza vacía).

Las cadenas de Neon y Atlas que también están en `pass.env` **no se usan dentro de Docker**: el
`docker-compose.yml` las reemplaza para que la API hable con los contenedores `db` y `mongo`.

In [3]:
from dotenv import dotenv_values

valores = dotenv_values(RAIZ / "pass.env")
for variable in ["SECRET_KEY", "DEMO_ADMIN_PASSWORD", "DEMO_MEDICO_PASSWORD",
                 "DEMO_EPS_PASSWORD", "DEMO_PACIENTE_PASSWORD"]:
    print(f"{variable:24s}", "OK" if valores.get(variable) else "FALTA  <- agrégala a pass.env")

SECRET_KEY               OK
DEMO_ADMIN_PASSWORD      OK
DEMO_MEDICO_PASSWORD     OK
DEMO_EPS_PASSWORD        OK
DEMO_PACIENTE_PASSWORD   OK


---
## 3. Levantar todo el proyecto

Un solo comando levanta los 7 servicios. `--build` construye la imagen de la API desde `python/Dockerfile`
(la primera vez tarda unos minutos porque descarga las imágenes de Postgres, Mongo, Orthanc, nginx y HAPI).

| Servicio | Imagen | Puerto en tu máquina | Para qué |
|---|---|---|---|
| `db` | postgres:16 | 5434 | pacientes, usuarios, remisiones, observaciones, historial, auditoría |
| `mongo` | mongo:7 | 27017 | bitácora `gestiones_contacto` |
| `api` | propia (`python/Dockerfile`) | 8000 | FastAPI: login, roles, CRUD, bloqueo |
| `front` | nginx:alpine | **8080** | interfaz gráfica; reparte `/api` → api y `/fhir` → HAPI |
| `orthanc` | orthancteam/orthanc | 8042 (solo tu equipo) | **PACS**: imágenes médicas DICOM; solo la API le habla |
| `hapi-fhir` | hapiproject/hapi | 8081 | servidor FHIR R4 |
| `fhir-db` | postgres:15 | 5433 | base propia de HAPI |

`db/schema.sql` está montado en `/docker-entrypoint-initdb.d`: se ejecuta solo, **únicamente la primera vez**
(cuando el volumen está vacío).

In [4]:
!docker compose -f "{COMPOSE}" up -d --build

#1 [internal] load local bake definitions
#1 reading from stdin 653B 0.0s done
#1 DONE 0.0s

#2 [internal] load build definition from Dockerfile
#2 transferring dockerfile: 798B 0.0s done
#2 DONE 0.0s

#3 [internal] load metadata for docker.io/library/python:3.11-slim
#3 DONE 1.1s

#4 [internal] load .dockerignore
#4 transferring context: 226B done
#4 DONE 0.0s

#5 [1/8] FROM docker.io/library/python:3.11-slim@sha256:e41613d42d4891e4930f79523f93f81bbc7632584ec65e36ab055f41a800b41e
#5 resolve docker.io/library/python:3.11-slim@sha256:e41613d42d4891e4930f79523f93f81bbc7632584ec65e36ab055f41a800b41e 0.0s done
#5 DONE 0.1s

#6 [internal] load build context
#6 transferring context: 909B done
#6 DONE 0.0s

#7 [5/8] COPY python/ ./python/
#7 CACHED

#8 [6/8] COPY frontend/ ./frontend/
#8 CACHED

#9 [7/8] COPY data/ ./data/
#9 CACHED

#10 [2/8] WORKDIR /app
#10 CACHED

#11 [4/8] RUN pip install --no-cache-dir -r requirements.txt
#11 CACHED

#12 [3/8] COPY requirements.txt .
#12 CACHED

#13 [8/

 Image docker-api Building 
 Image docker-api Built 
 Container trazared_orthanc Running 
 Container trazared_fhir_db Running 
 Container trazared_hapi_fhir Running 
 Container trazared_mongo Running 
 Container trazared_api Running 
 Container trazared_db Recreate 
 Container trazared_front Recreate 
 Container trazared_db Recreated 
 Container trazared_front Recreated 
 Container trazared_db Starting 
 Container trazared_db Started 
 Container trazared_db Waiting 
 Container trazared_mongo Waiting 
 Container trazared_mongo Healthy 
 Container trazared_db Healthy 
 Container trazared_front Starting 
 Container trazared_front Started 


In [5]:
!docker compose -f "{COMPOSE}" ps

NAME                 IMAGE                                                                     COMMAND                  SERVICE     CREATED         STATUS                   PORTS
trazared_api         sha256:ae8f4585a008f2139ccd060298629826574c26ddc5372ff7b8b51cfc57612c0f   "uvicorn main:app --…"   api         18 hours ago    Up 18 hours              0.0.0.0:8000->8000/tcp, [::]:8000->8000/tcp
trazared_db          postgres:16                                                               "docker-entrypoint.s…"   db          8 seconds ago   Up 6 seconds (healthy)   0.0.0.0:5434->5432/tcp, [::]:5434->5432/tcp
trazared_fhir_db     postgres:15                                                               "docker-entrypoint.s…"   fhir-db     18 hours ago    Up 18 hours              0.0.0.0:5433->5432/tcp, [::]:5433->5432/tcp
trazared_front       nginx:alpine                                                              "/docker-entrypoint.…"   front       8 seconds ago   Up Less than a second 

`STATUS` debe decir `Up` (y `healthy` en `db` y `mongo`) en los 7 contenedores. HAPI FHIR tarda 1–2 minutos más en quedar listo.
Si algo no sube, revisa los logs, por ejemplo: `!docker compose -f "{COMPOSE}" logs api --tail 50`

In [6]:
!docker compose -f "{COMPOSE}" logs api --tail 20

trazared_api  | INFO:     172.18.0.8:60376 - "GET /gestiones-contacto?remision_id=3 HTTP/1.1" 200 OK
trazared_api  | INFO:     172.18.0.8:33552 - "GET /pacientes/3/imagenes HTTP/1.1" 200 OK
trazared_api  | INFO:     172.18.0.8:33556 - "GET /imagenes/e49d4628-11c9cfbf-fbd971b5-4423d372-91cd6991/preview HTTP/1.1" 200 OK
trazared_api  | INFO:     172.18.0.8:46322 - "GET /remisiones?incluir_inactivas=true HTTP/1.1" 200 OK
trazared_api  | INFO:     172.18.0.8:46326 - "GET /remisiones/291 HTTP/1.1" 200 OK
trazared_api  | INFO:     172.18.0.8:46332 - "GET /pacientes/291 HTTP/1.1" 200 OK
trazared_api  | INFO:     172.18.0.8:46336 - "GET /gestiones-contacto?remision_id=291 HTTP/1.1" 200 OK
trazared_api  | INFO:     172.18.0.8:46334 - "GET /observaciones?remision_id=291 HTTP/1.1" 200 OK
trazared_api  | INFO:     172.18.0.8:35858 - "GET /health HTTP/1.1" 200 OK
trazared_api  | INFO:     172.18.0.8:35866 - "GET /health HTTP/1.1" 200 OK
trazared_api  | INFO:     172.18.0.8:43788 - "GET /health HTTP

In [7]:
import requests

BASE = "http://localhost:8000"          # API directa
FRONT = "http://localhost:8080"         # nginx (interfaz + /api + /fhir)

print("API /health        ->", requests.get(f"{BASE}/health", timeout=10).json())
print("nginx /api/health  ->", requests.get(f"{FRONT}/api/health", timeout=10).json())
try:
    r = requests.get(f"{FRONT}/fhir/metadata", timeout=30)
    print("nginx /fhir        ->", r.status_code, r.json().get("fhirVersion"))
except Exception as e:
    print("HAPI todavía está arrancando, vuelve a correr esta celda en un minuto:", e)

API /health        -> {'api': 'ok', 'postgres': 'ok', 'mongo': 'ok', 'pacs': 'ok'}
nginx /api/health  -> {'api': 'ok', 'postgres': 'ok', 'mongo': 'ok', 'pacs': 'ok'}
nginx /fhir        -> 200 4.0.1


---
## 4. La base de datos: columnas nuevas para el bloqueo

`schema.sql` solo **agrega** (es idempotente, también se puede correr contra Neon sin borrar nada):
- `usuarios.intentos_fallidos` (contador) y `usuarios.bloqueado` (bandera).
- `auditoria.detalle` (texto con el contexto) y `auditoria.registro_id` ahora admite vacío
  (un login fallido no apunta a ningún registro).
- `pacientes.fecha_nacimiento`, `telefono`, `tipo_sangre` y `alergias` (opcionales): los muestra la ficha del paciente de la interfaz.

In [8]:
!docker compose -f "{COMPOSE}" exec db psql -U trazared -d trazared -c "\dt"
!docker compose -f "{COMPOSE}" exec db psql -U trazared -d trazared -c "\d usuarios"
!docker compose -f "{COMPOSE}" exec db psql -U trazared -d trazared -c "\d auditoria"

                List of relations
 Schema |         Name         | Type  |  Owner   
--------+----------------------+-------+----------
 public | auditoria            | table | trazared
 public | observaciones        | table | trazared
 public | pacientes            | table | trazared
 public | remisiones           | table | trazared
 public | remisiones_historial | table | trazared
 public | usuarios             | table | trazared
(6 rows)

                                  Table "public.usuarios"
      Column       |  Type   | Collation | Nullable |               Default                
-------------------+---------+-----------+----------+--------------------------------------
 id                | integer |           | not null | nextval('usuarios_id_seq'::regclass)
 nombre            | text    |           | not null | 
 correo            | text    |           | not null | 
 contrasena_hash   | text    |           | not null | 
 rol               | text    |           | not null | 
 

In [9]:
# Usuarios que la API creó sola al arrancar (admin, médico y EPS)
!docker compose -f "{COMPOSE}" exec db psql -U trazared -d trazared -c "SELECT id, correo, rol, intentos_fallidos, bloqueado FROM usuarios ORDER BY id;"

 id |       correo        |   rol    | intentos_fallidos | bloqueado 
----+---------------------+----------+-------------------+-----------
  1 | admin@trazared.huv  | admin    |                 0 | f
  2 | medico@huv.gov.co   | medico   |                 0 | f
  3 | eps@coosalud.com    | eps      |                 0 | f
  4 | paciente@correo.com | paciente |                 0 | f
(4 rows)



---
## 5. Tarea 1 — Base de datos del proyecto final (dataset de remisiones)

`python/cargar_dataset.py` genera y carga **600 remisiones sintéticas** en tres grupos definidos por el resultado de
la remisión (igual que el ejemplo de la clase: diabéticos / sanos / borderline):

- **efectiva** (200): aceptada o completada, 1–2 contactos, casi siempre con convenio, triage casi normal.
- **fallida** (200): rechazada tras 4–7 contactos ("sin cupo en UCI", "no hay especialista"...), casi nunca con
  convenio, triage alterado.
- **borderline** (200): pendiente o en gestión, 3–4 contactos, convenio mixto, triage intermedio que se traslapa
  con los otros dos.

Se carga en las tablas que ya existen (`pacientes`, `remisiones`, `observaciones` con 7 signos vitales LOINC/UCUM) y en
MongoDB (`gestiones_contacto`, con la misma estructura de siempre). También crea el usuario de rol paciente.
El diccionario de datos completo está en `data/README.md`.

In [10]:
!docker compose -f "{COMPOSE}" exec api python cargar_dataset.py

[cargar] 0 remisiones nuevas cargadas (0 signos vitales, 0 bitácoras en MongoDB); 600 ya existían y se saltaron


In [11]:
# Verificación en PostgreSQL
!docker compose -f "{COMPOSE}" exec db psql -U trazared -d trazared -c "SELECT estado, COUNT(*) AS remisiones, ROUND(100.0*AVG(convenio_vigente::int),1) AS pct_convenio FROM remisiones GROUP BY estado ORDER BY estado;"
!docker compose -f "{COMPOSE}" exec db psql -U trazared -d trazared -c "SELECT codigo_loinc, tipo, unidad, ROUND(AVG(valor),1) AS promedio FROM observaciones GROUP BY 1,2,3 ORDER BY 1;"

   estado   | remisiones | pct_convenio 
------------+------------+--------------
 aceptada   |         93 |         83.9
 completada |        107 |         86.9
 en_gestion |        129 |         46.5
 pendiente  |         71 |         54.9
 rechazada  |        200 |         17.0
(5 rows)

 codigo_loinc |            tipo             | unidad  | promedio 
--------------+-----------------------------+---------+----------
 59408-5      | Saturación de oxígeno       | %       |     91.6
 8310-5       | Temperatura corporal        | Cel     |     37.6
 8462-4       | Presión arterial diastólica | mm[Hg]  |     68.0
 8480-6       | Presión arterial sistólica  | mm[Hg]  |    110.6
 8867-4       | Frecuencia cardiaca         | /min    |    103.4
 9269-2       | Escala de Glasgow           | {score} |     12.9
 9279-1       | Frecuencia respiratoria     | /min    |     22.2
(7 rows)



In [12]:
# Verificación en MongoDB (el contenedor publica el puerto 27017 en tu máquina)
import pymongo, pprint

gestiones = pymongo.MongoClient("mongodb://localhost:27017/")["trazared_huv"]["gestiones_contacto"]
print("Documentos en gestiones_contacto:", gestiones.count_documents({}))
ejemplo = gestiones.find_one({}, {"_id": 0, "remision_id": 1, "contactos": 1})
ejemplo["contactos"] = ejemplo["contactos"][:2]
pprint.pprint(ejemplo)

Documentos en gestiones_contacto: 600
{'contactos': [{'contactado_por': 'Enf. Luz Mery Rivas',
                'fecha': datetime.datetime(2026, 4, 20, 17, 32),
                'institucion_contactada': 'Clínica Farallones',
                'medio': 'telefono',
                'respuesta': 'Rechazan por no tener autorización de la EPS'},
               {'contactado_por': 'Enf. Luz Mery Rivas',
                'fecha': datetime.datetime(2026, 4, 20, 22, 19),
                'institucion_contactada': 'Clínica Nuestra Señora de los '
                                          'Remedios',
                'medio': 'telefono',
                'respuesta': 'No hay especialista disponible'}],
 'remision_id': 1}


In [14]:
%pip install pandas requests python-dotenv pymongo

   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.7 MB 1.5 MB/s eta 0:00:06
   --- ------------------------------------ 0.8/9.7 MB 1.5 MB/s eta 0:00:06
   ---- ----------------------------------- 1.0/9.7 MB 1.4 MB/s eta 0:00:06
   ----- ---------------------------------- 1.3/9.7 MB 1.6 MB/s eta 0:00:06
   ------- -------------------------------- 1.8/9.7 MB 1.7 MB/s eta 0:00:05
   --------- ------------------------------ 2.4/9.7 MB 1.8 MB/s eta 0:00:05
   ------------- -------------------------- 3.1/9.7 MB 2.0 MB/s eta 0:00:04
   --------------- ------------------------ 3.7/9.7 MB 2.1 MB/s eta 0:00:03
   ------------------ --------------------- 4.5/9.7 MB 2.2 MB/s eta 0:00:03
   --------------------- ------------------ 5.2/9.7 MB 2.4 MB/s eta 0:00:02
   ------------------------ -----

In [15]:
# La tabla lista para el análisis de la próxima clase: una fila por remisión
import pandas as pd

df = pd.read_csv(RAIZ / "data" / "dataset_analitico.csv")
print(df.shape)
df.groupby("grupo")[["fc", "pas", "spo2", "glasgow", "n_contactos", "horas_gestion", "convenio_vigente"]].mean().round(1)

(600, 22)


,fc,pas,spo2,glasgow,n_contactos,horas_gestion,convenio_vigente
grupo,,,,,,,
borderline,104.0,110.5,91.3,13.0,3.5,8.5,0.5
efectiva,88.5,126.8,95.4,14.6,1.4,0.4,0.9
fallida,117.7,94.6,88.2,11.0,5.5,19.9,0.2


---
## 5b. Base de imágenes médicas (PACS con Orthanc)

Igual que en el cuaderno de la clase: las imágenes **no** se guardan en PostgreSQL. Viven en **Orthanc** como archivos
**DICOM**, y se enlazan con nuestro paciente por el tag `PatientID` (0010,0020) = número de documento.

Reglas de seguridad (las mismas de la clase):
1. El navegador **nunca** le habla a Orthanc: todo pasa por la API (`/pacientes/{id}/imagenes`, `/imagenes/{id}/preview`),
   que verifica el token y el rol (solo admin y médico) y deja cada acceso en `auditoria` (`imagen_subida`, `imagen_vista`).
2. Al subir, la API abre el archivo con **Pillow** (no confía en la extensión), acepta solo PNG/JPEG de hasta 15 MB y lo
   **vuelve a codificar** para borrar metadatos ocultos (EXIF, GPS).
3. Orthanc tiene su propio usuario (`api` / `orthanc_dev_2026`) y su puerto solo se publica en tu equipo.

`python/sembrar_imagenes.py` crea la base de imágenes del proyecto: **fantomas sintéticos** (no son imágenes clínicas)
según el motivo de la remisión — radiografía de tórax para neumonía / insuficiencia respiratoria / politrauma /
cardiología, y TAC de cráneo para trauma craneoencefálico / hemorragia. Los pacientes del grupo *fallida* tienen hallazgos
más marcados, para que las imágenes queden alineadas con los datos. Salen con poco contraste a propósito: en el visor, al
subir el contraste, aparecen las estructuras.

In [16]:
!docker compose -f "{COMPOSE}" exec api python sembrar_imagenes.py --max 60

[imágenes] 0 creadas en el PACS; 60 pacientes ya tenían imágenes


In [17]:
# Verificación: el PACS responde y un paciente con imágenes las lista por la API
print("API /health ->", requests.get(f"{BASE}/health").json())

valores_env = dotenv_values(RAIZ / "pass.env")
r = requests.post(f"{BASE}/login", data={"username": "medico@huv.gov.co", "password": valores_env["DEMO_MEDICO_PASSWORD"]})
H_MED = {"Authorization": f"Bearer {r.json()['access_token']}"}
for p in requests.get(f"{BASE}/pacientes", headers=H_MED).json()[:80]:
    imgs = requests.get(f"{BASE}/pacientes/{p['id']}/imagenes", headers=H_MED).json()["imagenes"]
    if imgs:
        print(f"{p['nombre']} (CC {p['documento']}):", [(i["modalidad_nombre"], i["descripcion"]) for i in imgs])
        break

API /health -> {'api': 'ok', 'postgres': 'ok', 'mongo': 'ok', 'pacs': 'ok'}
Ana Torres Mina (CC 1175373460): [('Radiografía', 'Radiografía de tórax PA (sintética)')]


Abre también **http://localhost:8042** (usuario `api`, clave `orthanc_dev_2026`): es la interfaz propia de Orthanc, donde se
ven los mismos estudios como DICOM. En la interfaz de TrazaRed, abre la ficha de un paciente → pestaña **Imágenes** →
clic en una miniatura: se abre el **visor** (brillo, contraste, negativo, zoom con la rueda, arrastrar para mover y la
lectura del valor de cada píxel). La fórmula es la de la clase: `salida = contraste × (entrada − 128) + 128 + brillo`.

---
## 6. Tarea 2 — Bloqueo al 3er intento fallido

Reglas implementadas en `POST /login` (`python/main.py`):
1. Cada contraseña errada suma 1 a `intentos_fallidos` y responde **401** diciendo cuántos intentos quedan.
2. Al tercero, `bloqueado = TRUE` y responde **423 (Locked)**.
3. Bloqueado, no entra **ni con la contraseña correcta**.
4. Todo queda en `auditoria`: `login_fallido`, `bloqueo_usuario`, `login_bloqueado`, `login_exitoso`, `desbloqueo_usuario`.
5. Solo un admin desbloquea: `POST /usuarios/{id}/desbloquear` (o el botón **Desbloquear** en la interfaz).

⚠️ La prueba se hace con el **médico**, nunca con el admin (si el único admin se bloquea, nadie lo puede desbloquear
desde la API; ver el final de esta sección).

In [18]:
credenciales = {
    "admin": ("admin@trazared.huv", valores["DEMO_ADMIN_PASSWORD"]),
    "medico": ("medico@huv.gov.co", valores["DEMO_MEDICO_PASSWORD"]),
}

def login(correo, clave):
    r = requests.post(f"{BASE}/login", data={"username": correo, "password": clave})
    return r.status_code, r.json()

correo_medico, clave_medico = credenciales["medico"]
for i, clave in enumerate(["mala-1", "mala-2", "mala-3", clave_medico], start=1):
    status, cuerpo = login(correo_medico, clave)
    etiqueta = "contraseña CORRECTA" if clave == clave_medico else "contraseña mala"
    print(f"Intento {i} ({etiqueta}): {status} -> {cuerpo['detail']['mensaje'] if 'detail' in cuerpo else 'token recibido'}")

Intento 1 (contraseña mala): 401 -> Contraseña incorrecta. Te quedan 2 intentos antes del bloqueo.
Intento 2 (contraseña mala): 401 -> Contraseña incorrecta. Te queda 1 intento antes del bloqueo.
Intento 3 (contraseña mala): 423 -> Usuario bloqueado por intentos fallidos. Solo un administrador puede desbloquearlo.
Intento 4 (contraseña CORRECTA): 423 -> Usuario bloqueado por intentos fallidos. Solo un administrador puede desbloquearlo.


In [19]:
# El admin ve al médico bloqueado y lo desbloquea
status, cuerpo = login(*credenciales["admin"])
H_ADMIN = {"Authorization": f"Bearer {cuerpo['access_token']}"}

bloqueados = requests.get(f"{BASE}/usuarios", params={"solo_bloqueados": True}, headers=H_ADMIN).json()
print("Bloqueados:", [(u["id"], u["correo"], u["intentos_fallidos"]) for u in bloqueados])

for u in bloqueados:
    r = requests.post(f"{BASE}/usuarios/{u['id']}/desbloquear", headers=H_ADMIN)
    print("Desbloquear", u["correo"], "->", r.status_code, "| bloqueado:", r.json()["bloqueado"])

status, cuerpo = login(correo_medico, clave_medico)
print("Login del médico después del desbloqueo ->", status, "token recibido" if status == 200 else cuerpo)

Bloqueados: [(2, 'medico@huv.gov.co', 3)]
Desbloquear medico@huv.gov.co -> 200 | bloqueado: False
Login del médico después del desbloqueo -> 200 token recibido


In [20]:
# Evidencia: la auditoría (más reciente primero, hora de Colombia)
eventos = requests.get(f"{BASE}/auditoria", params={"limite": 10}, headers=H_ADMIN).json()
pd.DataFrame(eventos)[["fecha", "correo", "accion", "detalle"]]

,fecha,correo,accion,detalle
0,2026-09-29T07:03:32.449047-05:00,medico@huv.gov.co,login_exitoso,medico@huv.gov.co
1,2026-09-29T07:03:32.152898-05:00,admin@trazared.huv,desbloqueo_usuario,medico@huv.gov.co desbloqueado por admin@traza...
2,2026-09-29T07:03:32.116341-05:00,admin@trazared.huv,login_exitoso,admin@trazared.huv
3,2026-09-29T07:03:23.440266-05:00,medico@huv.gov.co,login_bloqueado,medico@huv.gov.co: intento de ingreso con el u...
4,2026-09-29T07:03:23.408713-05:00,medico@huv.gov.co,bloqueo_usuario,medico@huv.gov.co: bloqueado al llegar a 3 int...
5,2026-09-29T07:03:23.404444-05:00,medico@huv.gov.co,login_fallido,medico@huv.gov.co: contraseña incorrecta (inte...
6,2026-09-29T07:03:23.142785-05:00,medico@huv.gov.co,login_fallido,medico@huv.gov.co: contraseña incorrecta (inte...
7,2026-09-29T07:03:22.890890-05:00,medico@huv.gov.co,login_fallido,medico@huv.gov.co: contraseña incorrecta (inte...
8,2026-09-29T07:03:13.479592-05:00,medico@huv.gov.co,login_exitoso,medico@huv.gov.co
9,2026-09-28T12:55:58.769957-05:00,admin@trazared.huv,imagen_vista,instancia e49d4628-11c9cfbf-fbd971b5-4423d372-...


In [21]:
# Lo mismo, directo en la tabla
!docker compose -f "{COMPOSE}" exec db psql -U trazared -d trazared -c "SELECT id, usuario_id, accion, registro_id, detalle, fecha FROM auditoria ORDER BY id DESC LIMIT 10;"

 id | usuario_id |       accion       | registro_id |                                detalle                                 |           fecha            
----+------------+--------------------+-------------+------------------------------------------------------------------------+----------------------------
 11 |          2 | login_exitoso      |           2 | medico@huv.gov.co                                                      | 2026-09-29 07:03:32.449047
 10 |          1 | desbloqueo_usuario |           2 | medico@huv.gov.co desbloqueado por admin@trazared.huv                  | 2026-09-29 07:03:32.152898
  9 |          1 | login_exitoso      |           1 | admin@trazared.huv                                                     | 2026-09-29 07:03:32.116341
  8 |          2 | login_bloqueado    |           2 | medico@huv.gov.co: intento de ingreso con el usuario bloqueado         | 2026-09-29 07:03:23.440266
  7 |          2 | bloqueo_usuario    |           2 | medico@huv.gov.co: b

**Si el admin llega a quedar bloqueado** (no hay otro admin que lo desbloquee), se hace directo en la base:
```sql
UPDATE usuarios SET bloqueado = FALSE, intentos_fallidos = 0 WHERE correo = 'admin@trazared.huv';
```
En Docker: `!docker compose -f "{COMPOSE}" exec db psql -U trazared -d trazared -c "UPDATE usuarios SET bloqueado = FALSE, intentos_fallidos = 0 WHERE correo = 'admin@trazared.huv';"`
(en Neon, desde su SQL Editor).

---
## 7. La interfaz gráfica — http://localhost:8080

Abre **http://localhost:8080**. Es una SPA (una sola página que cambia de vista sin recargar), con el diseño de referencia
del equipo: login dividido, menú lateral, saludo, ficha del paciente con pestañas y línea de estado de la remisión.

| Rol | Usuario | Qué ve |
|---|---|---|
| Admin | `admin@trazared.huv` | Inicio, Remisiones (incluidas las eliminadas, puede restaurar), Pacientes, Hospitales de destino, Reportes, **Usuarios** (botón *Desbloquear*) y **Auditoría** |
| Médico | `medico@huv.gov.co` | Inicio, Mis remisiones (crear, editar y eliminar las suyas), Pacientes, Hospitales, Reportes; en la ficha: signos vitales, gestiones de contacto e **imágenes (PACS)** |
| EPS | `eps@coosalud.com` | Inicio, remisiones de sus afiliados (solo lectura), Hospitales y Reportes de sus remisiones |
| Paciente | `paciente@correo.com` | Mi remisión: sus datos, a dónde va y la línea de estado del traslado |

La ficha tiene las pestañas **Datos del paciente**, **Información clínica** (signos vitales LOINC), **Datos de la remisión**,
**Gestiones** (bitácora de MongoDB) e **Imágenes** (PACS + visor), y a la derecha el **hospital de destino** y el **estado
de la remisión** (Solicitud enviada → En proceso → Confirmación → Traslado).

En el login: mensaje "te quedan N intentos" con barras, aviso rojo de usuario bloqueado, mostrar/ocultar contraseña,
"Recordar usuario" e indicadores de conexión (API, PostgreSQL, MongoDB, PACS, FHIR). Si quieren una foto en el panel
izquierdo del login (como en el diseño de referencia), guárdenla como `frontend/img/login.jpg`.

---
## 8. Al terminar: detener o eliminar

Igual que en la clase, de menos a más agresivo:
- `stop`: apaga los contenedores sin borrar nada (se retoma con `start`).
- `down`: borra los contenedores; los datos siguen en los volúmenes (`pgdata`, `mongodata`, `fhir_db_data`).
- `down -v`: borra también los volúmenes (incluidas las imágenes de Orthanc) → la próxima vez arranca con la base vacía (y `schema.sql` se vuelve a ejecutar).

In [22]:
# Opción 1: solo detener
!docker compose -f "{COMPOSE}" stop

 Container trazared_front Stopping 
 Container trazared_front Stopped 
 Container trazared_api Stopping 
 Container trazared_hapi_fhir Stopping 
 Container trazared_api Stopped 
 Container trazared_db Stopping 
 Container trazared_mongo Stopping 
 Container trazared_orthanc Stopping 
 Container trazared_db Stopped 
 Container trazared_mongo Stopped 
 Container trazared_hapi_fhir Stopped 
 Container trazared_fhir_db Stopping 
 Container trazared_fhir_db Stopped 
 Container trazared_orthanc Stopped 


In [23]:
#Opción 2: eliminar contenedores (los datos se conservan)
!docker compose -f "{COMPOSE}" down

 Container trazared_front Stopping 
 Container trazared_front Stopped 
 Container trazared_front Removing 
 Container trazared_front Removed 
 Container trazared_hapi_fhir Stopping 
 Container trazared_api Stopping 
 Container trazared_hapi_fhir Stopped 
 Container trazared_hapi_fhir Removing 
 Container trazared_api Stopped 
 Container trazared_api Removing 
 Container trazared_hapi_fhir Removed 
 Container trazared_fhir_db Stopping 
 Container trazared_fhir_db Stopped 
 Container trazared_fhir_db Removing 
 Container trazared_api Removed 
 Container trazared_db Stopping 
 Container trazared_mongo Stopping 
 Container trazared_orthanc Stopping 
 Container trazared_db Stopped 
 Container trazared_db Removing 
 Container trazared_mongo Stopped 
 Container trazared_mongo Removing 
 Container trazared_orthanc Stopped 
 Container trazared_orthanc Removing 
 Container trazared_fhir_db Removed 
 Container trazared_db Removed 
 Container trazared_orthanc Removed 
 Container trazared_mongo Rem

In [24]:
# Opción 3: eliminar contenedores Y datos (empezar de cero)
!docker compose -f "{COMPOSE}" down -v

 Volume docker_fhir_db_data Removing 
 Volume docker_orthanc_data Removing 
 Volume docker_mongodata Removing 
 Volume docker_pgdata Removing 
 Volume docker_fhir_db_data Removed 
 Volume docker_orthanc_data Removed 
 Volume docker_mongodata Removed 
 Volume docker_pgdata Removed 
